# Cross-Cloud, Hybrid, and Edge Strategies
Companion notebook for the **Cross-Cloud, Hybrid, and Edge Strategies** section of Chapter 12.

Place each component where its data, latency, and reliability needs are actually served -- not everywhere at once. This notebook loads the real `placement.yaml` shipped alongside it, the book's example unchanged.

In [1]:
print(open("placement.yaml").read())

placement:
  cloud:
    - api-gateway
    - planner
    - global-observability
  on_prem:
    - erp-connector
    - private-document-index
  edge:
    - speech-to-text
    - camera-event-filter



In [2]:
import yaml

placement = yaml.safe_load(open("placement.yaml"))["placement"]
for tier, components in placement.items():
    print(f"{tier}: {components}")

cloud: ['api-gateway', 'planner', 'global-observability']
on_prem: ['erp-connector', 'private-document-index']
edge: ['speech-to-text', 'camera-event-filter']


The book doesn't give code for *choosing* a tier, only the resulting config -- here's a small decision function built from the tradeoffs it describes: cross-cloud for resilience, hybrid for private-system access, edge for latency or local operation:

In [3]:
def recommend_tier(*, needs_low_latency: bool, must_stay_on_prem: bool, needs_resilience: bool) -> str:
    if must_stay_on_prem:
        return "on_prem"
    if needs_low_latency:
        return "edge"
    if needs_resilience:
        return "cloud"  # a second-region/second-provider cloud placement
    return "cloud"


components = {
    "camera-event-filter": {"needs_low_latency": True, "must_stay_on_prem": False, "needs_resilience": False},
    "erp-connector": {"needs_low_latency": False, "must_stay_on_prem": True, "needs_resilience": False},
    "global-observability": {"needs_low_latency": False, "must_stay_on_prem": False, "needs_resilience": True},
}

for name, needs in components.items():
    print(f"{name:22s} -> {recommend_tier(**needs)}")

camera-event-filter    -> edge
erp-connector          -> on_prem
global-observability   -> cloud


Every one of those matches the shipped `placement.yaml` exactly -- `camera-event-filter` under `edge`, `erp-connector` under `on_prem`, `global-observability` under `cloud`.

**Sync delay and partial failure** are the operational cost of splitting placement -- queue work when a link is slow or down, and make retries idempotent so a retried event never gets applied twice:

In [4]:
processed_event_ids: set[str] = set()


def apply_edge_event(event_id: str, payload: dict) -> str:
    """Idempotent: replaying the same event_id after a dropped link is a no-op, not a duplicate."""
    if event_id in processed_event_ids:
        return "already_applied"
    processed_event_ids.add(event_id)
    return "applied"


print(apply_edge_event("evt-501", {"stock_delta": -1}))
print(apply_edge_event("evt-501", {"stock_delta": -1}))  # retried after a flaky link

applied
already_applied
